In [1]:
import pandas as pd
import glob
import os

# --- Config ---
# Run this script from inside the `variables` directory
# i.e. cd .../DRIMS/data/variables && python concatenate_all_variables.py
INPUT_DIR   = "."
OUTPUT_PATH = "drims_master.csv"
# --------------

# Step 1: Discover all subfolders
folders = sorted([
    d for d in os.listdir(INPUT_DIR)
    if os.path.isdir(os.path.join(INPUT_DIR, d)) and not d.startswith(".")
])

print(f"Found {len(folders)} variable folders.\n")

# Step 2: For each folder, read all CSVs, tag with date from filename, concat into one df per variable
variable_dfs = {}

for folder in folders:
    folder_path = os.path.join(INPUT_DIR, folder)
    files = sorted(glob.glob(os.path.join(folder_path, "*.csv")))

    if not files:
        print(f"  [SKIP] {folder} — no CSVs found")
        continue

    dfs = []
    for f in files:
        basename = os.path.basename(f).replace(".csv", "")  # e.g. Animal_Affected_Big_2021_06
        parts = basename.split("_")
        year, month = parts[-2], parts[-1]                  # year and month from filename

        try:
            df = pd.read_csv(f)
        except Exception as e:
            print(f"  [ERROR] Could not read {f}: {e}")
            continue

        # Normalise object_id column name
        df.columns = [c.strip() for c in df.columns]
        id_col = next((c for c in df.columns if c.lower() in ("object_id", "objectid")), None)
        if id_col is None:
            print(f"  [SKIP] {f} — no object_id column (columns: {list(df.columns)})")
            continue
        if id_col != "object_id":
            df.rename(columns={id_col: "object_id"}, inplace=True)

        df["date"] = f"{year}_{month}"   # e.g. 2021_06 — taken from filename
        dfs.append(df)

    if not dfs:
        print(f"  [SKIP] {folder} — all files skipped")
        continue

    combined = pd.concat(dfs, ignore_index=True)

    # Rename value columns with folder prefix to avoid clashes
    value_cols = [c for c in combined.columns if c not in ("object_id", "date")]
    combined.rename(columns={c: f"{folder}__{c}" for c in value_cols}, inplace=True)

    variable_dfs[folder] = combined
    print(f"  Loaded {folder}: {len(combined)} rows, {len(files)} files")

print(f"\nLoaded {len(variable_dfs)} variables. Merging...\n")

# Step 3: Merge all variables on object_id + date
base_key = ["object_id", "date"]
dfs_list = list(variable_dfs.values())

master = dfs_list[0]
for df in dfs_list[1:]:
    master = master.merge(df, on=base_key, how="outer")

master.sort_values(["object_id", "date"], inplace=True)
master.reset_index(drop=True, inplace=True)

# Step 4: Save
master.to_csv(OUTPUT_PATH, index=False)
print(f"Done. Master sheet: {len(master)} rows x {len(master.columns)} columns → {OUTPUT_PATH}")

Found 50 variable folders.

  Loaded Animal_Affected_Big: 2528 rows, 18 files
  Loaded Animal_Affected_Poultry: 2528 rows, 18 files
  Loaded Animal_Affected_Small: 2528 rows, 18 files
  Loaded Animal_Washed_Away_Big: 2528 rows, 18 files
  Loaded Animal_Washed_Away_Poultry: 2528 rows, 18 files
  Loaded Animal_Washed_Away_Small: 2528 rows, 18 files
  Loaded Bridge: 1311 rows, 30 files
  Loaded Children_Camp: 2528 rows, 18 files
  Loaded Crop_Area: 1467 rows, 31 files
  Loaded Embankment breached: 1311 rows, 30 files
  Loaded Embankments affected: 1315 rows, 33 files
  Loaded Female_Camp: 2528 rows, 18 files
  Loaded House_Damaged_Others_CattleShed: 2528 rows, 18 files
  Loaded House_Damaged_Others_Huts: 2528 rows, 18 files
  Loaded House_Fully_Damaged_Kuccha: 2528 rows, 18 files
  Loaded House_Fully_Damaged_Pukka: 2528 rows, 18 files
  Loaded House_Partially_Damaged_Kuccha: 2528 rows, 18 files
  Loaded House_Partially_Damaged_Pukka: 2528 rows, 18 files
  Loaded Human_Live_Lost: 1311 rows